
# Supervised Classification / Regression Modeling & Tuning

This notebook trains and compares four supervised classification models on the Titanic dataset.
It includes:
- Leak-free preprocessing using a Scikit-Learn `ColumnTransformer` + `Pipeline`
- Four baseline models
- Hyperparameter tuning using `GridSearchCV` with stratified 5-fold cross-validation
- Precision, Recall, F1-score, ROC-AUC and confusion matrices
- ROC-AUC curves for model comparison
- Selection of a model based on validation F1-score, with ROC-AUC used as a secondary metric
- Saving the selected model as a `.joblib` file


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42


## 1. Load the dataset

In [ ]:

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
display(df.head())


## 2. Prepare target and train/test split

The split is performed before fitting preprocessing steps to reduce the risk of data leakage.

In [ ]:

df = df.drop(columns=["PassengerId", "Name", "Ticket"], errors="ignore")

X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)


## 3. Build the preprocessing pipeline

In [ ]:

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


## 4. Define four baseline model architectures

In [ ]:

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}

baseline_results = []

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]

    baseline_results.append({
        "Model": name,
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

baseline_results_df = pd.DataFrame(baseline_results).sort_values("F1", ascending=False)
display(baseline_results_df.round(4))


## 5. Hyperparameter tuning with stratified 5-fold GridSearchCV

In [ ]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1, 10],
        "model__solver": ["liblinear"]
    },
    "Decision Tree": {
        "model__max_depth": [3, 5, 7, None],
        "model__min_samples_split": [2, 5, 10]
    },
    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 5, 10],
        "model__min_samples_split": [2, 5]
    },
    "Gradient Boosting": {
        "model__n_estimators": [50, 100],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3]
    }
}

tuned_models = {}
tuning_summary = []

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        refit=True
    )

    grid.fit(X_train, y_train)
    tuned_models[name] = grid.best_estimator_

    tuning_summary.append({
        "Model": name,
        "Best CV F1": grid.best_score_,
        "Best Parameters": grid.best_params_
    })

tuning_summary_df = pd.DataFrame(tuning_summary).sort_values("Best CV F1", ascending=False)
display(tuning_summary_df)


## 6. Evaluate tuned models on the held-out test set

In [ ]:

evaluation_results = []
predictions = {}
probabilities = {}

for name, model in tuned_models.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    predictions[name] = pred
    probabilities[name] = prob

    evaluation_results.append({
        "Model": name,
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

evaluation_df = pd.DataFrame(evaluation_results).sort_values(
    ["F1", "ROC-AUC"], ascending=False
).reset_index(drop=True)

display(evaluation_df.round(4))


## 7. Confusion matrices and classification reports

In [ ]:

for name in tuned_models:
    print("=" * 70)
    print(name)
    print(classification_report(y_test, predictions[name]))

    cm = confusion_matrix(y_test, predictions[name])
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cbar=False)
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


## 8. ROC-AUC curves

In [ ]:

plt.figure(figsize=(8, 6))

for name, prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-AUC Comparison")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Select the model based on validation metrics

In [ ]:

# Primary criterion: F1-score on the held-out test set.
# Secondary criterion: ROC-AUC.
champion_name = evaluation_df.iloc[0]["Model"]
champion_model = tuned_models[champion_name]

print("Selected model:", champion_name)
print("Selection criteria: highest F1-score, with ROC-AUC as the secondary metric.")
display(evaluation_df.iloc[[0]].round(4))


## 10. Save the selected model

In [ ]:

model_filename = "champion_model.joblib"
joblib.dump(champion_model, model_filename)

print(f"Saved selected model to: {model_filename}")



## Summary

- Four supervised classification architectures were trained: Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting.
- Preprocessing was kept inside Scikit-Learn pipelines so transformations are learned from training folds only.
- Hyperparameters were optimized using `GridSearchCV` with stratified 5-fold cross-validation.
- Models were evaluated using Precision, Recall, F1-score, ROC-AUC, classification reports, and confusion matrices.
- ROC-AUC curves were plotted for comparison.
- The selected model was saved as `champion_model.joblib`.

> Note: Model selection in this notebook is based on the documented evaluation metrics and is not a claim that the model is universally superior for every use case.
